In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import f1_score
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [13]:
print(f"Cargando datos...")
dataset_train = pd.read_json("./data/dataset_polaridad_es_train_embeddings.json", lines=True)
dataset_test = pd.read_json("./data/dataset_polaridad_es_test_embeddings.json", lines=True)


Cargando datos...


In [15]:
le = LabelEncoder()

# El texto ya está en su forma vectorial calculado por los word embeddings del archivo según el campo
# Campo = Vector de Word Embeddings de 300 dimensiones
# we_ft = Word Embeddings calculados de textos generales del español 
# we_mx = Word Embeddings calculados de textos del español de México
# we_es = Word Embeddings calculados de textos del español de España

X_trainext = dataset_train['text'].to_numpy()
            # Convertir a matriz de arrays de numpy
X_es = np.vstack(dataset_train["we_es"].to_numpy())
X_mx = np.vstack(dataset_train["we_mx"].to_numpy())
X_ft = np.vstack(dataset_train["we_ft"].to_numpy())
Y_train = dataset_train['klass'].to_numpy()

X_train = np.concatenate([X_es, X_mx, X_ft], axis=1) 

X_test_text = dataset_test['text'].to_numpy()
# Convertir a matriz de arrays de numpy
X_es_t = np.vstack(dataset_test["we_es"].to_numpy())
X_mx_t = np.vstack(dataset_test["we_mx"].to_numpy())
X_ft_t = np.vstack(dataset_test["we_ft"].to_numpy())
X_test = np.concatenate([X_es_t, X_mx_t, X_ft_t], axis=1) 
Y_test = dataset_test['klass'].to_numpy()


In [16]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Y_train_encoded= le.fit_transform(Y_train)
Y_test_encoded= le.transform(Y_test)

In [20]:
total_features = X_train.shape[1] # 900
n_comp = int(total_features * 0.30)
pca = PCA(n_components=n_comp)

X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)


In [21]:

# clf = svm.LinearSVC() = svm.SVC(kernel='linear')  son equivalentes los dos modelos.  
# kernel{‘linear’, ‘poly’, ‘rbf’, ‘sigmoid’}
clf = svm.SVC(kernel='rbf')
# clf = svm.SVC(kernel='rbf')

# Entrenar el modelo con los word embeddings
clf.fit(X_train, Y_train_encoded)
# Predecir los datos con el modelo entrenado para el conjunto de test
y_pred = clf.predict(X_test)
# Evaluar el desempeño de acuerdo a la métrica f1-macro
score = f1_score(Y_test_encoded, y_pred, average="macro")
print(f"Desempeño del modelo en el conjunto de test: {score}")

Desempeño del modelo en el conjunto de test: 0.6528238723785825


In [23]:
from sklearn.metrics import classification_report
# TODO: Evaluar el desempeño del modelo 
print(classification_report(Y_test_encoded, y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.6636    0.4220    0.5159       173
           1     0.7000    0.8491    0.7674       371
           2     0.6991    0.6529    0.6752       242

    accuracy                         0.6947       786
   macro avg     0.6876    0.6413    0.6528       786
weighted avg     0.6917    0.6947    0.6836       786

